In [9]:
import pandas as pd #  pandas
import seaborn as sns # seaborn
import matplotlib.pyplot as plt # matplotlib.pyplot
from pprint import pprint # pprint para visualización metadatos
from ucimlrepo import fetch_ucirepo # se importa fetch_ucilrepo
import numpy as np # nump
import seaborn as snst # seaborn
import statsmodels.api as sm #statsmodels
from sklearn.model_selection import train_test_split # train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder # StandarScaler y LabelEncoder
from sklearn.linear_model import LogisticRegression # LR
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # CR, CM y ACC score
from sklearn.model_selection import GridSearchCV # grid search
from sklearn.tree import DecisionTreeClassifier # DR
from sklearn.ensemble import RandomForestClassifier # RF
from sklearn.neighbors import KNeighborsClassifier # kNN
import os # os

In [ ]:
# Se carga el dataset DARWIN
darwin = fetch_ucirepo(id=732)
# X (features)
x = darwin.data.features

# Y (targets)
y = darwin.data.targets

# Se concatena features y target en un mismo dataframe 'df'
df = pd.concat([x,y], axis=1)

In [10]:
# Implementación base de datos alternativa
# ruta
ruta_darwin = os.path.expanduser("/Users/joseromerodegaetano/Desktop/DARWIN/DARWIN.csv") 

try:

    # Se carga el dataset
    df = pd.read_csv(ruta_darwin)

    x = df.drop(columns=['class'])
    y = df['class']

    print('Dataset DARWIN cargado correctamente')

except FileNotFoundError:
    print(f"No se encuentra el archivo .csv DARWIN en {ruta_darwin}")

Dataset DARWIN cargado correctamente


# PRUEBA ELIMINACIÓN DE VARIABLES

En este documento se repiten los análisis de ML eliminando las varaibles relacionadas, que presentan una alta correlación ( r > 0.90), para estudiar una posible mejora del modelo.

Datos de correlación obtenidos del heatmpad correspondiente a la tarea 19

Las variables que se descartan por su correlación son:
- `total_time` - `air_time`; r = 1.00
- `gmrt_in_air` - `mean_speed_in_air`; r = 0.99
- `gmrt_on_paper` - `mean_speed_on_paper`; r = 0.96
- `mean_jerk_in_air` - `mean_acc_in_air`; r = 1.00

In [11]:
# Se eliminan las variables con alta correlacion (r > 0.9)
var_eliminar = ['total_time', 'gmrt_in_air', 'gmrt_on_paper', 'mean_jerk_in_air'] # variables a eliminar

# Lista completa con las columnas a eliminar
var_eliminar_final = [
    f"{var}{i}"
    for var in var_eliminar
    for i in range(1,26)
]

# Se eliminan las columnas del df original
df2 = df.drop(columns=var_eliminar_final, errors='ignore')

# Se verifica la correcta eliminación
print(f"Columnas eliminadas: {len(var_eliminar_final)}")
print(f"Nuevo tamaño del df: {df2.shape}")

Columnas eliminadas: 100
Nuevo tamaño del df: (174, 352)


## Preparación de los datos

In [12]:
# Se elimina la columna ID
# X = df (sin 'ID' ni 'class'), axis = 1 para cols
X = df2.drop(['ID', 'class'], axis = 1)

# Y contiene 'class'
Y = df2['class']

# Se codficia paciente (P) -> 1 y sano (H) -> 0
le = LabelEncoder()
Y = le.fit_transform(Y)

# Se dividen los datos en 80% para train y 20% para test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Se revisa la correcta división de los datos
print(f"Datos de entrenamiento {len(X_train)}")
print(f"Datos test {len(X_test)}")

Datos de entrenamiento 139
Datos test 35


## Estandarización de los datos

In [13]:
# StandardScaler()
scaler = StandardScaler()

# Se ajusta la estandarización con X_train
X_train_scaled = scaler.fit_transform(X_train)

# Se aplica la transformación al test
X_test_scaled = scaler.transform(X_test)

## 1. Regresión logística (LR)

### 1.1. Entrenamiento del modelo

Se entrena el modelo con los datos de entrenamiento (`X_train_scaled` y `Y_train`)

In [14]:
# Se crea el modelo
model_LR = LogisticRegression(max_iter=1000)

# Se entrena el modelo
model_LR.fit(X_train_scaled, Y_train)

# Se realizan las predicciones
Y_pred = model_LR.predict(X_test_scaled)

# Informe de clasificación
print("--- INFORME DE CLASIFICACIÓN ---")
print(classification_report(Y_test, Y_pred))

# Matriz de confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(Y_test, Y_pred))

# Matriz
tn, fp, fn, tp = confusion_matrix(Y_test, Y_pred).ravel()

# Métricas 
# Sensibilidad (capacidad para detectar los pacientes correctamente)
sensitivity = tp / (tp + fn)

# Especificdad (capacidad para dectectar los sanos correctamente)
specificity = tn / (tn + fp)

print(f"Sensibilidad: {sensitivity:.2f}")
print(f"Especificidad: {specificity:.2f}")

--- INFORME DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

           0       0.69      0.65      0.67        17
           1       0.68      0.72      0.70        18

    accuracy                           0.69        35
   macro avg       0.69      0.68      0.68        35
weighted avg       0.69      0.69      0.69        35

--- MATRIZ DE CONFUSIÓN ---
[[11  6]
 [ 5 13]]
Sensibilidad: 0.72
Especificidad: 0.65


**Conclusiones:** los datos obtenidos para este modelo de RL coinciden con los obtenidos al emplear todas las variables, al ser esta exploración incial, y al no ser un resultado concluyente, para la aplicación inicial de los algoritmos de machine learning (ML), se mantendrán todas las variables.